## 1. 한 줄 진단

좌표 변화는 거의 잡았지만, **`(r, c) -> (c, n-1-r)`라는 일반 공식과 “덮어쓰기 없이 제자리에서 바꾸는 방법”으로 연결하지 못한 게 핵심 실패 원인**이다.

---

## 2. 내 사고 흐름 요약

처음에는 문제 조건을 잘 정리했다.
정사각형 `n x n`, 90도 시계방향 회전, 그리고 **새 matrix를 만들면 안 되고 기존 matrix를 직접 수정해야 한다**는 핵심 조건도 인식했다. 이 조건은 LeetCode 원문 요구사항과도 맞다. ([LeetCode][1])

그다음 3x3 예시에서 각 좌표가 어디로 이동하는지 직접 나열했다.

```text
0,0 -> 0,2
0,1 -> 1,2
0,2 -> 2,2
...
```

여기까지는 좋았다. 하지만 좌표 나열 이후에 **공식화** 또는 **제자리 swap 전략**으로 넘어가지 못하고 멈췄다.

---

## 3. 막힌 이유 분석

**첫 오판:**
“열이 행이 되고 행이 열이 된다”까지만 잡고, 거기서 끝났다.

정확히는 단순히 `행 <-> 열`만 바뀌는 게 아니라,

```text
기존 (r, c) -> 회전 후 (c, n - 1 - r)
```

이 된다.

예를 들어 `3x3`에서 `(2, 0)`은:

```text
(r, c) = (2, 0)
(c, n - 1 - r) = (0, 3 - 1 - 2) = (0, 0)
```

네 메모의 `2,0 -> 0,0`과 일치한다.

**결정적으로 부족했던 점:**
좌표 변화를 관찰했지만, 그걸 **일반화된 인덱스 공식**으로 압축하지 못했다. 그리고 설령 공식을 알았더라도, 이 문제는 새 배열을 만들 수 없기 때문에 `matrix[c][n-1-r] = matrix[r][c]`처럼 바로 대입하면 기존 값이 덮어써지는 문제가 생긴다.

**왜 여기서 막혔는지:**
이 문제는 “좌표 공식”만으로 끝나는 문제가 아니라, 그 공식을 **in-place 변환 방식**으로 바꿔야 한다. 여기서 필요한 발상은 보통 두 가지다.

첫째, `상하 반전 + 주대각선 전치`로 90도 회전을 만든다.
둘째, 테두리별로 4개 좌표를 한 묶음으로 잡고 순환 swap한다.

표준 풀이 중 하나는 matrix를 위아래로 뒤집고, 그다음 주대각선 기준으로 전치하면 90도 시계방향 회전이 된다는 방식이다. 이 방식도 결국 `(i, j) -> (j, n - i - 1)` 이동을 만들어낸다. ([GitHub][2])

---

## 4. 실패 유형 분류

**주 실패 유형:**
4. 상태/불변식 설계 실패

**부 실패 유형:**
3. 패턴 인식 실패

**근거:**
문제 해석은 거의 맞았다. 조건도 잘 읽었고, 좌표 변화도 직접 추적했다. 그래서 문제 해석 실패는 아니다.

막힌 핵심은 “좌표가 이렇게 이동한다”에서 “그럼 어떤 swap 단위로 안전하게 바꿀 것인가?”로 넘어가지 못한 것이다. 즉, 방향은 맞았지만 제자리 회전의 불변식을 설계하지 못했다.

---

## 5. 등급 판정

**판정: D**

**이유:**
좌표 추적까지는 자력으로 접근했지만, 정답 구조인 `전치 + 반전` 또는 `4-way swap`으로 이어지지 못했다. 해설을 보면 이해할 가능성은 높지만, 현재 메모만 보면 백지 상태에서 구현까지 자력으로 밀어붙이기는 어려운 단계다.

**유사 유형 경험 점검:**
네가 과거에 풀던 문제들은 hash map, stack, greedy, sliding window처럼 “자료구조 선택”이 중심인 문제가 많았다. 이 문제는 그보다 **2차원 배열의 좌표 변환**과 **in-place swap 순서**가 중심이다. 그래서 기존 패턴과 바로 연결하기 어려웠을 가능성이 크다.

처음 만난 신호는 이거다.

```text
2D matrix + 회전/뒤집기 + in-place
```

이 신호가 나오면 단순 순회보다 먼저 **좌표 변환 공식**과 **swap 순서**를 생각해야 한다.

---

## 6. 정답 풀이에서 뽑아낼 일반화 포인트

### 자료구조/알고리즘 포인트

이 문제는 BFS, DP, hash map 문제가 아니다. 핵심은 **2차원 배열 좌표 변환 + in-place swap**이다.

떠올릴 수 있는 대표 풀이 패턴은 두 개다.

첫 번째는:

```text
상하 반전 -> 주대각선 전치
```

두 번째는:

```text
layer별로 4개 좌표를 순환 swap
```

네 메모는 “가장 바깥 테두리”를 언급했기 때문에 두 번째 풀이 쪽으로 접근하고 있었다. 다만 테두리의 네 점을 한 번에 묶는 식까지 못 간 것이다.

예를 들어 한 위치 `(r, c)`를 기준으로 4개 좌표는 이런 식으로 순환한다.

```text
top-left     = (r, c)
top-right    = (c, n - 1 - r)
bottom-right = (n - 1 - r, n - 1 - c)
bottom-left  = (n - 1 - c, r)
```

하지만 이 방식은 인덱스가 헷갈리기 쉽다. 그래서 처음 익힐 때는 `상하 반전 + 전치`가 더 안정적이다.

---

### 상태/불변식 포인트

`상하 반전 + 전치` 풀이에서 유지해야 하는 생각은 이거다.

```text
90도 시계방향 회전은 한 번에 하려고 하지 말고,
두 개의 단순한 대칭 변환으로 쪼갤 수 있다.
```

예시:

```text
원본
1 2 3
4 5 6
7 8 9
```

상하 반전:

```text
7 8 9
4 5 6
1 2 3
```

주대각선 전치:

```text
7 4 1
8 5 2
9 6 3
```

이게 90도 시계방향 회전 결과다.

여기서 불변식은:

```text
상하 반전은 행의 순서만 바꾼다.
전치는 matrix[i][j]와 matrix[j][i]를 바꾼다.
두 작업 모두 기존 matrix 안에서 swap만 하므로 O(1) 추가 공간을 유지한다.
```

---

### 복잡도/경계조건 포인트

시간복잡도는 `O(n^2)`이다. matrix의 모든 칸을 최대 상수 번 정도 만지기 때문이다.

추가 공간은 `O(1)`이다. 새 2차원 배열을 만들지 않고 swap용 임시 변수 정도만 쓰기 때문이다. 문제에서도 input matrix를 직접 수정해야 하고 다른 2D matrix를 만들면 안 된다고 요구한다. ([LeetCode][1])

경계조건은 다음 정도만 조심하면 된다.

```text
n = 1이면 그대로 둬도 정답
전치할 때 i, j 전체를 다 돌면 같은 쌍을 두 번 swap할 수 있음
따라서 전치는 보통 j를 i+1부터 시작하거나 i부터 시작해도 자기 자신 swap만 허용
```

---

### 일반화 문장

**2D 배열 회전 문제는 좌표를 직접 대입하려 하지 말고, “좌표 변환 공식” 또는 “대칭 변환 조합”으로 바꿔서 in-place swap 가능한 형태를 먼저 찾아라.**

---

## 7. 다음에 써먹을 트리거 문장

`정사각형 matrix + 90도 회전 + in-place -> 전치 + 행/열 반전 점검`

`좌표가 (r, c)에서 규칙적으로 이동함 -> (r, c) -> 새 좌표 공식 먼저 세우기`

`값을 옮겨야 하는데 새 배열 금지 -> 직접 대입 금지, swap cycle 또는 대칭 변환으로 처리`

---

## 8. 개선 액션

**오늘 바로 할 것 1개:**
3x3 matrix를 손으로 두 번 변환해봐라.

```text
원본 -> 상하 반전 -> 전치
```

이 과정을 직접 쓰면서 왜 90도 회전이 되는지 확인하면 된다.

**내일 복습할 것 1개:**
아래 두 공식을 다시 적어봐라.

```text
90도 시계방향: (r, c) -> (c, n - 1 - r)
90도 반시계방향: (r, c) -> (n - 1 - c, r)
```

**비슷한 문제에서 확인할 포인트 1개:**
2D matrix 변환 문제가 나오면 바로 코드부터 쓰지 말고, 먼저 작은 예시에서 좌표 3~4개를 추적한 뒤 **공식화 가능한지** 확인해라.

---

## 9. 오답노트용 한 줄 요약

**D / 상태·불변식 설계 실패 / 좌표 이동은 봤지만 `(r,c)->(c,n-1-r)`와 in-place swap 전략으로 연결 못함 / “matrix 회전 + in-place -> 전치+반전 or 4-way swap” / 3x3 예시로 상하반전→전치 복습.**



주어진 매트릭스를 90도 시계방향으로 회전한 결과를 반환
입력형태는 이차원 배열

정사각형

n == matrix.length == matrix[i].length
1 <= n <= 20
-1000 <= matrix[i][j] <= 1000

매트릭스를 직접 수정해야함, 다른 매트릭스를 새로 만들어선 안됨

가장 바깥 테두리는 열이 행이되고 행이 열이됨

좌표상의 변화를 함 볼까

3*3 기준으로 
0,0 -> 0,2
0,1 -> 1,2
0,2 -> 2,2

1,0 -> 0,1
1,1 -> 1,1 (가운데 중심)
1,2 -> 2,1

2,0 -> 0,0
2,1 -> 1,0
2,2 -> 2,0

봐도 뭔가 공식을 모르겠다